# Project Analysis Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_GITHUB_USERNAME/YOUR_REPO_NAME/blob/main/notebooks/project_analysis_demo/analysis.ipynb)

本 Notebook 是一个演示笔记本，展示了如何：
1. **双端环境兼容**（本地与 Google Colab 动态挂载 Drive）
2. **分项目局部依赖加载**（在 Colab 下利用 **uv** 极速自动安装该子目录的 `requirements.txt`）
3. **引用共享的复用模块**（`src/utils.py`）
4. **规范的数据读写**（使用 `utils.get_data_path`）

## 步骤 1：环境初始化与局部依赖安装
当您在 Google Colab 中打开此 Notebook 时，以下单元格将自动挂载 Google Drive，并使用 `uv` 极速安装该子项目所特有的依赖库（如 `pandas` 等）。

In [ ]:
import sys
import os

# 1. 检测是否处于 Colab 环境
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # 挂载 Google Drive
    from google.colab import drive
    drive.mount('/content/drive')
    
    # 【注意】在此处修改为您在 Google Drive 中的 Git 仓库文件夹名称
    REPO_NAME = "notebooks" 
    REPO_PATH = f"/content/drive/MyDrive/{REPO_NAME}"
    
    if os.path.exists(REPO_PATH):
        os.chdir(REPO_PATH)
        print(f"工作路径已切换到 Google Drive 仓库: {REPO_PATH}")
    else:
        print(f"警告: Google Drive 中未找到 {REPO_PATH}。请确保已在此处克隆仓库。")
        
    # 安装 uv 并极速安装当前子项目的专属依赖到 Colab 全局系统中
    # 因为本笔记本在 notebooks/project_analysis_demo/ 目录下
    %cd {REPO_PATH}/notebooks/project_analysis_demo
    !pip install uv
    !uv pip install --system -r requirements.txt
else:
    print("当前为本地开发环境，无需挂载 Drive 与自动安装依赖。")

## 步骤 2：添加项目根目录至系统路径 (sys.path)
因为我们的通用工具模块定义在仓库根目录下的 `src/` 中，为了能直接在 Notebook 里 `import src.utils`，我们需要确保根目录已加入到 `sys.path` 中。

In [ ]:
# 寻找项目根目录并加入 sys.path
current_dir = os.path.abspath(os.getcwd())
root_dir = current_dir
for _ in range(5):
    # 向上寻找含有 .git 或 requirements.txt 的根目录
    if os.path.exists(os.path.join(root_dir, 'requirements.txt')):
        break
    root_dir = os.path.dirname(root_dir)

if root_dir not in sys.path:
    sys.path.append(root_dir)
    print(f"项目根目录已添加至系统路径: {root_dir}")

# 现在我们可以直接引用全局 src 中的工具了！
from src.utils import get_data_path, setup_colab
print("成功导入 src.utils!")

## 步骤 3：数据读取与分析
我们将展示如何通过 `get_data_path()` 读写数据。大文件由 Google Drive 存储，小文件由 Git 追踪，在代码里都用一致的函数处理路径。

In [ ]:
import pandas as pd
import numpy as np

# 1. 获取（或自动创建）数据路径下的文件路径
csv_filepath = get_data_path("demo_dataset.csv")
print(f"数据文件存放路径: {csv_filepath}")

# 2. 如果文件不存在，我们生成一份模拟数据存入此路径
if not os.path.exists(csv_filepath):
    df_mock = pd.DataFrame({
        'Date': pd.date_range(start='2026-01-01', periods=100),
        'Category': np.random.choice(['A', 'B', 'C'], size=100),
        'Value': np.random.randn(100).cumsum()
    })
    df_mock.to_csv(csv_filepath, index=False)
    print("未检测到历史数据，已在数据目录下生成模拟数据 'demo_dataset.csv'。")

# 3. 使用 Pandas 加载数据
df = pd.read_csv(csv_filepath)
print(f"成功读取数据！样本数据前 5 行：")
df.head()

In [ ]:
# 4. 简单的数据看板绘制
import matplotlib.pyplot as plt
%matplotlib inline

plt.figure(figsize=(10, 5))
plt.plot(df['Date'], df['Value'], label='Cumulative Value', color='#10b981', linewidth=2)
plt.title('Analysis Demo Metrics')
plt.xlabel('Date')
plt.ylabel('Value')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.show()